In [1]:
!pip install gpytorch torch
!pip install emcee
!pip install h5py

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
torch.set_default_dtype(torch.double)
import gpytorch
import pandas as pd

/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
# Load in simulation parameters and normalize
simulation_parameters = np.loadtxt("LHS_simulation_parameters_retrain8.csv", delimiter=',') #############
scaler_params = StandardScaler()
simulation_parameters_scaled = scaler_params.fit_transform(simulation_parameters)
X = simulation_parameters_scaled.astype(np.float32)


# Load in simulation output data
time_sim = np.linspace(0, 16, 385)
weeks = np.array([0,1,2,4,6,8,10,12,14,16], dtype=np.float64)
#weeks = np.array([0,1,4,16]) #np.arange(0, 17, 1)                      # 0..16
time_idx = [np.argmin(np.abs(time_sim - w)) for w in weeks]   # map to 385-grid

fibroblast_density = np.loadtxt("LHS_fibroblast_density_retrain8.csv", delimiter=',') ##############
collagen_density = np.loadtxt("LHS_collagen_density_retrain8.csv", delimiter=',') #############
oxygen_tension = np.loadtxt("LHS_oxygen_tension_retrain8.csv", delimiter=',') #############
capillary_density = np.loadtxt("LHS_capillary_density_retrain8.csv", delimiter=',') ############
macrophage_density = np.loadtxt("LHS_macrophage_density_retrain8.csv", delimiter=',') ##########
AF_conc = np.loadtxt("LHS_AF_conc_retrain8.csv", delimiter=',') ############


species_list = [
    fibroblast_density,    # (N, 385)
    collagen_density,      # (N, 385) 
    oxygen_tension,        # (N, 385)
    capillary_density,     # (N, 385)
    macrophage_density,    # (N, 385)
    AF_conc                # (N, 385)
]




num_species = len(species_list)   
N, D = X.shape
T = len(time_idx)                 
Q = num_species * T  


# Build a (N, 6, 17) cube of targets, then flatten to 102 columns
Y_cube = np.stack([S[:, time_idx] for S in species_list], axis=1).astype(np.float32)
Y_mat  = Y_cube.reshape(N, Q)



X_train = simulation_parameters_scaled.copy()
Y_train= Y_mat.copy()

LHS_retraining = pd.read_csv("LHS_Samples_Retraining7.csv", header= None, index_col= 0) #########################
X_test_raw = LHS_retraining.values.transpose()

# ---- Apply same scaling as training ----
X_test = scaler_params.transform(X_test_raw)


from sklearn.preprocessing import StandardScaler

# ---------- STANDARDIZE TARGETS PER SPECIES (fit on TRAIN only) ----------
scalers_Y   = [None] * num_species
Y_train_std = Y_train.copy()

for s in range(num_species):
    cols = slice(s*T, (s+1)*T)  # block of T columns for species s
    sc = StandardScaler(with_mean=True, with_std=True)
    Y_train_std[:, cols] = sc.fit_transform(Y_train[:, cols])
    scalers_Y[s] = sc

# ---------- TORCH TENSORS ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Xtr = torch.from_numpy(X_train.astype(np.float64)).to(device)
Ytr = torch.from_numpy(Y_train_std.astype(np.float64)).to(device)
Xte = torch.from_numpy(X_test.astype(np.float64)).to(device)

# ---------- MODEL ----------
lengthscale_orig = np.array([
    0.01, 0.05, 2.5, 1000, 5e-13, 5e-06, 5e-12, 5e-7, 0.0005,
    5000, 0.005, 1000, 0.001, 5e-8
], dtype=float)
lengthscale_std = (lengthscale_orig / scaler_params.scale_).astype(np.float64)




# ===== Full-Cholesky Species × Convolutional RBF(Time) MTGP (per-species time lengthscales) =====
# Expects you already have: Xtr, Ytr, device, Q (num_tasks), D (input dim),
#                           num_species (S), weeks (len T array), T
import torch
import torch.nn.functional as F
import gpytorch
from linear_operator import to_linear_operator

# ---------- Full-cholesky species kernel (exact free-form SPD; S(S+1)/2 params) ----------
class FullCholeskySpeciesKernel(gpytorch.kernels.Kernel):
    has_lengthscale = False

    def __init__(self, num_species: int, init_diag: float = 1.0):
        super().__init__()
        self.S = int(num_species)

        invsp = gpytorch.utils.transforms.inv_softplus
        L_raw = torch.zeros(self.S, self.S, dtype=torch.get_default_dtype())
        with torch.no_grad():
            for i in range(self.S):
                L_raw[i, i] = invsp(torch.tensor(init_diag, dtype=torch.get_default_dtype()))
        self.register_parameter("L_raw", torch.nn.Parameter(L_raw))

    @property
    def covar_matrix(self):
        # Build lower-tri L with positive diag via softplus
        L = torch.tril(self.L_raw)
        diag_soft = F.softplus(torch.diag(L)) + 1e-6
        L = L - torch.diag(torch.diag(L)) + torch.diag(diag_soft)
        return to_linear_operator(L @ L.T)   # (S x S) Lazy PSD


# ---------- Species × Time task kernel with CONVOLUTIONAL per-species time lengthscales ----------
class SpeciesTimeTaskKernel_Conv(gpytorch.kernels.Kernel):
    """
    Builds a (S*T) x (S*T) task covariance by blocks:
      K_task[(i,t),(j,t')] = K_species[i,j] * K_conv_time^{(i,j)}(|t - t'|)
    where K_conv_time^{(i,j)} is the SE×SE convolution:
      coeff = 2*ℓ_i*ℓ_j / (ℓ_i^2 + ℓ_j^2)
      K     = coeff * exp( - (Δ^2) / (ℓ_i^2 + ℓ_j^2) )
    """
    has_lengthscale = False  # task kernel (no X-lengthscale)

    def __init__(self, num_species, weeks, T, init_time_lengthscale=1.0):
        super().__init__()
        self.S = int(num_species)
        self.T = int(T)

        # Species coreg: free-form SPD via full Cholesky
        self.species_kernel = FullCholeskySpeciesKernel(self.S, init_diag=1.0)

        # Week grid (T x 1) and precompute pairwise squared deltas Δ^2
        weeks_t = torch.as_tensor(weeks, dtype=torch.get_default_dtype()).view(-1, 1)
        self.register_buffer("weeks", weeks_t)
        with torch.no_grad():
            delta = weeks_t - weeks_t.t()                 # (T x T)
            self.register_buffer("delta_sq", delta.pow(2))  # (T x T)

        # Per-species positive time lengthscales ℓ_i (init to 1.0 per paper)
        # Parameterize via softplus: ℓ = softplus(raw) to keep > 0
        raw_ell = torch.log(torch.exp(torch.tensor(float(init_time_lengthscale))) - 1.0)
        self.raw_ell = torch.nn.Parameter(raw_ell * torch.ones(self.S, dtype=weeks_t.dtype))

        # Optional global outputscale for time part (start at 1.0)
        self.log_time_outputscale = torch.nn.Parameter(torch.zeros((), dtype=weeks_t.dtype))

    def _ell(self):
        return F.softplus(self.raw_ell) + 1e-8            # (S,)

    @property
    def covar_matrix(self):
        # Species covariance matrix (S x S) as dense tensor
        Ks_lazy = self.species_kernel.covar_matrix
        Ks = Ks_lazy.to_dense()  # small S, ok to materialize

        # Per-species ℓ and global time amplitude
        ell = self._ell()                      # (S,)
        t_amp = self.log_time_outputscale.exp()

        # Build full (S*T) x (S*T) block matrix
        T = self.T
        delta_sq = self.delta_sq              # (T x T)
        K = Ks.new_zeros(self.S * T, self.S * T)

        for i in range(self.S):
            ℓi2 = ell[i] * ell[i]
            for j in range(self.S):
                ℓj2 = ell[j] * ell[j]
                denom = (ℓi2 + ℓj2).clamp_min(1e-12)
                coeff = torch.sqrt((2.0 * ell[i] * ell[j]) / denom) * t_amp
                # Convolutional SE block in time between species i and j
                Kt_ij = coeff * torch.exp(- delta_sq / denom)  # (T x T)
                # Multiply by species covariance scalar Ks[i,j]
                block = Ks[i, j] * Kt_ij
                # Place in big matrix
                r0, r1 = i * T, (i + 1) * T
                c0, c1 = j * T, (j + 1) * T
                K[r0:r1, c0:c1] = block

        return to_linear_operator(K)  # (S*T x S*T) Lazy operator


# ---------- Model: (RBF over X) ⊗ (Species×Time (convolutional) task kernel) ----------
class MTExactGP_FullSpeciesTime_Conv(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood,
                 num_species, weeks, T, num_tasks, ard_dims,
                 icm_rank=1):
        super().__init__(train_x, train_y, likelihood)

        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.ZeroMean(), num_tasks=num_tasks
        )

        # RQ with ARD, wrapped in a ScaleKernel (amplitude)
        self.data_kernel = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RQKernel(ard_num_dims=ard_dims)
        )

        self.covar_module = gpytorch.kernels.MultitaskKernel(
            self.data_kernel, num_tasks=num_tasks, rank=icm_rank
        )
        self.covar_module.task_covar_module = SpeciesTimeTaskKernel_Conv(
            num_species=num_species, weeks=weeks, T=T, init_time_lengthscale=1.0
        )

    def forward(self, x):
        mean_x  = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)


t_min, t_max = weeks.min(), weeks.max()             # 0 and 16
weeks_scaled = (weeks - t_min) / (t_max - t_min).astype(np.float64)
# ===================== Build likelihood + model =====================
# ===================== Build likelihood + model =====================
from gpytorch.constraints import GreaterThan

# 1) Create ONE likelihood, add floor, init small (per-task)
likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=Q).to(device)
likelihood.register_constraint("raw_task_noises", GreaterThan(1e-6))
with torch.no_grad():
    likelihood.task_noises.copy_(torch.full_like(likelihood.task_noises, 1e-2))  # 0.01 each

# 2) Build model with THAT likelihood
model = MTExactGP_FullSpeciesTime_Conv(
    Xtr, Ytr, likelihood,
    num_species=num_species, weeks=weeks_scaled, T=T,
    num_tasks=Q, ard_dims=D,
    icm_rank=1
).to(device)

print("task_covar_module:", type(model.covar_module.task_covar_module).__name__)

# 3) Init ARD + amplitudes + time amp
with torch.no_grad():
    # ARD: set RQ lengthscales (note: now under .base_kernel)
    model.data_kernel.base_kernel.lengthscale.copy_(
        torch.from_numpy(lengthscale_std).to(device)
    )
    # amplitude for RQ
    model.data_kernel.outputscale.fill_(1.0)

    # time kernel amp
    model.covar_module.task_covar_module.log_time_outputscale.zero_()


# (optional) freeze noises during warmup, then unfreeze later
freeze_noise_steps = 75
for p in likelihood.parameters():
    p.requires_grad_(False)


# ===================== Train =====================
# ===================== TRAIN (GPy-like: Adam warmup + LBFGS) =====================

from gpytorch.constraints import GreaterThan

# Use float64 like GPy
torch.set_default_dtype(torch.double)

model.train(); likelihood.train()

# ---- (Optional) seed ARD lengthscales for Kx with your standardized priors ----
# (uncomment if you have `lengthscale_std` on CPU/GPU already)
# with torch.no_grad():
#     model.data_kernel.lengthscale.copy_(
#         torch.as_tensor(lengthscale_std, dtype=torch.double, device=Xtr.device)
#     )
#     model.data_kernel.raw_outputscale.fill_(0.0)  # outputscale = 1.0

# Ensure multitask noise has a floor (GPy-ish lower bound)
likelihood.register_constraint("raw_task_noises", GreaterThan(1e-6))


# Exact MLL (ML-II)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

# ---------- 1) Short Adam warm-up ----------
adam_steps = 1200
if adam_steps > 0:
    adam = torch.optim.Adam(model.parameters(), lr=0.02)
    for i in range(1, adam_steps + 1):
        adam.zero_grad()
        with gpytorch.settings.cholesky_jitter(1e-5), gpytorch.settings.max_cholesky_size(10_000):
            output = model(Xtr)
            loss = -mll(output, Ytr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        adam.step()
        if i == freeze_noise_steps:
            for p in likelihood.parameters():
                p.requires_grad_(True)
        if i % 20 == 0:
            print(f"[Adam {i:03d}] loss={loss.item():.6f}, noise={likelihood.task_noises.mean().item():.6g}")

# ---------- 2) L-BFGS polish (like GPy) ----------
lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=0.2,                 # try 0.2–1.0 if needed
    max_iter=800,
    history_size=50,
    line_search_fn='strong_wolfe',
)

def closure():
    lbfgs.zero_grad()
    with gpytorch.settings.cholesky_jitter(1e-5), gpytorch.settings.max_cholesky_size(10_000):
        output = model(Xtr)
        loss = -mll(output, Ytr)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    return loss

final_loss = lbfgs.step(closure)


print(f"[LBFGS] final loss={final_loss.item():.6f}, noise={likelihood.task_noises.mean().item():.6g}")
# ===================== END TRAIN =====================

task_covar_module: SpeciesTimeTaskKernel_Conv
[Adam 020] loss=1.260925, noise=0.693148
[Adam 040] loss=1.243630, noise=0.693148
[Adam 060] loss=1.236706, noise=0.693148
[Adam 080] loss=1.211214, noise=0.644407
[Adam 100] loss=1.097412, noise=0.476119
[Adam 120] loss=0.983395, noise=0.356895
[Adam 140] loss=0.869780, noise=0.285338
[Adam 160] loss=0.758397, noise=0.246344
[Adam 180] loss=0.651521, noise=0.224666
[Adam 200] loss=0.550764, noise=0.211983
[Adam 220] loss=0.457882, noise=0.205883
[Adam 240] loss=0.374432, noise=0.204391
[Adam 260] loss=0.300918, noise=0.206062
[Adam 280] loss=0.236565, noise=0.209767
[Adam 300] loss=0.180248, noise=0.214194
[Adam 320] loss=0.130805, noise=0.218168
[Adam 340] loss=0.087021, noise=0.220772
[Adam 360] loss=0.047625, noise=0.22124
[Adam 380] loss=0.011580, noise=0.219159
[Adam 400] loss=-0.020856, noise=0.216319
[Adam 420] loss=-0.049339, noise=0.215995
[Adam 440] loss=-0.074588, noise=0.217049
[Adam 460] loss=-0.096964, noise=0.21752
[Adam 480

In [4]:
import numpy as np
from scipy.stats import gaussian_kde

# ------------------------------------------------------------
# 1) Load previous posterior samples (physical space)
# ------------------------------------------------------------
data = np.load("mcmc_samples_logprob12_full.npz")
samples_phys = data["samples_phys"]   # shape (Ns, D)

# Your physical-space bounds (14 x 2)
param_bounds = np.array([
    [1E-3, 7E-2],        # d_w
    [0.005, 0.1],        # p_m
    [0.5, 5],            # p_mc
    [100, 2000],         # K_mb
    [5e-13, 1e-11],      # p_vrho
    [3.5E-5, 7.0E-5],    # K_vrho
    [1e-12, 1e-10],      # p_vm
    [5E-7, 7.5E-6],      # d_vb
    [0.0005, 0.05],      # d_v
    [62.6793, 12472.9067], # d_bphi
    [0.005, 0.05],       # p_b
    [9640, 19280],       # K_bb
    [0.015, 0.020],      # p_rhoc
    [5E-8, 3.633E-7]     # p_phic
], dtype=float)

# ------------------------------------------------------------
# 2) Convert posterior samples to SCALED space (same scaler as MCMC uses)
# ------------------------------------------------------------
samples_scaled = scaler_params.transform(samples_phys)  # (Ns, D)

# ------------------------------------------------------------
# 3) Fit 1D KDE per-parameter (independent KDE prior) ONCE
# ------------------------------------------------------------
kdes_scaled = []
for j in range(samples_scaled.shape[1]):
    s = samples_scaled[:, j]
    s = s[np.isfinite(s)]
    if s.size < 10:
        raise ValueError(f"Too few finite samples to fit KDE in dim {j}: n={s.size}")
    kdes_scaled.append(gaussian_kde(s))

# ------------------------------------------------------------
# 4) Keep your scaled bounds as hard support
# ------------------------------------------------------------
bounds_phys   = np.vstack([param_bounds[:, 0], param_bounds[:, 1]])  # (2, D)
bounds_scaled = scaler_params.transform(bounds_phys)                  # (2, D)

low_scaled  = np.minimum(bounds_scaled[0], bounds_scaled[1])
high_scaled = np.maximum(bounds_scaled[0], bounds_scaled[1])

# ------------------------------------------------------------
# 5) Precompute per-dimension logpdf tables (LOOKUP TABLE)
# ------------------------------------------------------------
LOG_FLOOR = 1e-300  # prevents log(0)

def build_kde_logpdf_tables(
    samples_scaled, kdes_scaled, low_scaled, high_scaled,
    n_grid=2048,
    q_lo=0.001, q_hi=0.999,
    pad_frac=0.10
):
    """
    Build per-dimension grids + log-density arrays for fast interpolation.
    Grid range is quantile-focused (better accuracy per grid point) and clipped
    to hard bounds in scaled space.
    Returns:
      grid_z[j]      : (n_grid,) grid points (scaled)
      grid_logpdf[j] : (n_grid,) log p_j(grid_z[j])
    """
    D = samples_scaled.shape[1]
    grid_z = []
    grid_logpdf = []

    for j in range(D):
        s = samples_scaled[:, j]
        s = s[np.isfinite(s)]

        lo = np.quantile(s, q_lo)
        hi = np.quantile(s, q_hi)

        width = hi - lo
        if not np.isfinite(width) or width <= 0:
            width = (high_scaled[j] - low_scaled[j])
            if not np.isfinite(width) or width <= 0:
                width = 1.0

        lo -= pad_frac * width
        hi += pad_frac * width

        # clip to hard support bounds
        lo = max(lo, low_scaled[j])
        hi = min(hi, high_scaled[j])

        # fallback if degenerate
        if hi <= lo:
            lo, hi = low_scaled[j], high_scaled[j]
            if hi <= lo:
                raise ValueError(f"Degenerate scaled bounds in dim {j}: [{lo}, {hi}]")

        z = np.linspace(lo, hi, int(n_grid))
        pdf = kdes_scaled[j](z)                # expensive, but ONCE
        pdf = np.maximum(pdf, LOG_FLOOR)
        lnp = np.log(pdf)

        grid_z.append(z)
        grid_logpdf.append(lnp)

    return grid_z, grid_logpdf

# Build the lookup tables ONCE
import time
t0 = time.time()
print("tick")
grid_z, grid_logpdf = build_kde_logpdf_tables(
    samples_scaled=samples_scaled,
    kdes_scaled=kdes_scaled,
    low_scaled=low_scaled,
    high_scaled=high_scaled,
    n_grid=2048  # try 4096 if you want smoother tails
)
print(f"KDE lookup table build time: {time.time() - t0:.2f} s")

tick
KDE lookup table build time: 600.39 s


In [5]:
# Species order you provided:
# 0=fibroblast, 1=collagen, 2=oxygen, 3=capillary, 4=macrophage, 5=AF
IDX_FIB = 0
IDX_COL = 1
IDX_OXY = 2
IDX_CAP = 3
IDX_MAC = 4
IDX_AF  = 5

from scipy.stats import norm
# Map week value -> index in your weeks array
week_to_idx = {int(w): i for i, w in enumerate(weeks)}

def predict_all_species_timeseries(theta_phys_1d):
    """
    ONE GP forward pass for a single parameter vector (physical units).
    Returns:
      mu_phys_all  : (num_species, T)
      var_phys_all : (num_species, T)
    """
    # scale inputs like training
    theta_scaled = scaler_params.transform(theta_phys_1d.reshape(1, -1)).astype(np.float64)

    model.eval(); likelihood.eval()
    with torch.no_grad():
        xb   = torch.from_numpy(theta_scaled).to(next(model.parameters()).device, dtype=torch.double)
        pred = likelihood(model(xb))
        mu_std  = pred.mean.detach().cpu().numpy().reshape(-1)     # (Q,)
        var_std = pred.variance.detach().cpu().numpy().reshape(-1) # (Q,)

    # reshape standardized prediction to (num_species, T)
    mu_std_cube  = mu_std.reshape(num_species, T).copy()
    var_std_cube = var_std.reshape(num_species, T).copy()

    # un-standardize per species block
    mu_phys_all  = np.zeros_like(mu_std_cube,  dtype=np.float64)
    var_phys_all = np.zeros_like(var_std_cube, dtype=np.float64)
    for s in range(num_species):
        mean_s = scalers_Y[s].mean_.astype(np.float64)   # (T,)
        std_s  = scalers_Y[s].scale_.astype(np.float64)  # (T,)
        mu_phys_all[s, :]  = mu_std_cube[s, :] * std_s + mean_s
        var_phys_all[s, :] = var_std_cube[s, :] * (std_s ** 2)

    return mu_phys_all, var_phys_all


def predict_all_species_timeseries_batch(theta_phys_batch):
    """
    Batched GP: theta_phys_batch shape = (N, D).
    Returns:
      mu_phys : (N, num_species, T)
      var_phys: (N, num_species, T)
    """
    # scale like training
    theta_scaled = scaler_params.transform(theta_phys_batch).astype(np.float64)

    model.eval(); likelihood.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        xb   = torch.from_numpy(theta_scaled).to(device, dtype=torch.double)   # (N, D)
        pred = likelihood(model(xb))                                           # multitask: (N, Q)
        mu_std  = pred.mean.detach().cpu().numpy().reshape(-1, Q)              # (N, Q)
        var_std = pred.variance.detach().cpu().numpy().reshape(-1, Q)          # (N, Q)

    # reshape to (N, S, T)
    mu_std_all  = mu_std.reshape(-1, num_species, T)
    var_std_all = var_std.reshape(-1, num_species, T)

    # unstandardize per species
    mu_phys  = np.empty_like(mu_std_all,  dtype=np.float64)
    var_phys = np.empty_like(var_std_all, dtype=np.float64)
    for s in range(num_species):
        mean_s = scalers_Y[s].mean_.astype(np.float64)[None, :]   # (1, T)
        std_s  = scalers_Y[s].scale_.astype(np.float64)[None, :]  # (1, T)
        mu_phys[:, s, :]  = mu_std_all[:, s, :] * std_s + mean_s
        var_phys[:, s, :] = var_std_all[:, s, :] * (std_s ** 2)

    return mu_phys, var_phys


# ---- Shared: observed weeks & indices ----
w_obs   = [1, 4, 16]
idx_obs = [week_to_idx[w] for w in w_obs]  # already defined earlier

# =========================
# Fibroblast (weeks 1,4,16)
# =========================
y_fib_obs = np.array([0.0, 377504.0, 215893.0], dtype=float)
y_fib_obs_std = np.array([5000, 94279, 45150], dtype=float)       # <-- fill with experimental SDs


# =========================
# Collagen (weeks 1,4,16)
# =========================
y_col_obs = np.array([0.0, 1.35, 2.33], dtype=float)
y_col_obs_std = np.array([0.05, 0.25, 0.35], dtype=float)       # <-- fill with experimental SDs



# =========================
# Macrophage (weeks 1,4,16)
# =========================
y_mac_obs = np.array([0.0, 8618.77, 8025.32], dtype=float)
y_mac_obs_std = np.array([250, 3415.02, 5332.20], dtype=float)       # <-- fill with experimental SDs



# =========================
# Capillary (weeks 1,4,16)
# =========================
y_cap_obs = np.array([0.0, 3905.68, 4550.41], dtype=float)
y_cap_obs_std = np.array([250, 1465.39, 1959.27], dtype=float)       # <-- fill with experimental SDs



def log_likelihood_total_batch(theta_phys_batch):
    """
    Same as your original likelihood, but with normalization factors per species
    to balance contributions across different magnitudes (e.g., fibroblast vs collagen).

    NEW: adds peak-time ordering constraints using one-sided Normal CDF:
      macrophage peaks before fibroblast
      macrophage peaks before capillary
      AF peaks before capillary
    """

    # --- User-defined normalization constants ---
    norm_fib = 55051.0
    norm_col = 1.0
    norm_mac = 3557.0
    norm_cap = 1929.0
    norm_oxy = 52.0
    norm_af  = 3.5e-6

    # Peak-order softness (in weeks). Smaller = stricter.
    sigma_peak = 0.5

    # ONE (batched) GP call for all walkers in this batch
    mu_all, var_all = predict_all_species_timeseries_batch(theta_phys_batch)  # (N,S,T)
    N = mu_all.shape[0]
    ll = np.zeros(N, dtype=float)
    obs = np.array(idx_obs, dtype=int)
    eps = 1e-12

    # ---- Normalize predictions and variances ----
    def apply_scale(spec_idx, scale):
        mu_all[:, spec_idx, :]  /= scale
        var_all[:, spec_idx, :] /= (scale**2)

    apply_scale(IDX_FIB, norm_fib)
    apply_scale(IDX_COL, norm_col)
    apply_scale(IDX_MAC, norm_mac)
    apply_scale(IDX_CAP, norm_cap)
    apply_scale(IDX_OXY, norm_oxy)
    apply_scale(IDX_AF,  norm_af)

    # ---- Normalize observed data and SDs ----
    y_fib_obs_N     = y_fib_obs     / norm_fib
    y_fib_obs_std_N = y_fib_obs_std / norm_fib

    y_col_obs_N     = y_col_obs     / norm_col
    y_col_obs_std_N = y_col_obs_std / norm_col

    y_mac_obs_N     = y_mac_obs     / norm_mac
    y_mac_obs_std_N = y_mac_obs_std / norm_mac

    y_cap_obs_N     = y_cap_obs     / norm_cap
    y_cap_obs_std_N = y_cap_obs_std / norm_cap

    OXY_MIN_W1_N = 5.0  / norm_oxy
    OXY_CAP_N    = 52.0 / norm_oxy

    # ---- Helper for Gaussian log-likelihood ----
    def gauss_ll(y_obs, y_sd, mu, var):
        sigma2 = np.clip(var + (y_sd[None, :]**2), 1e-12, None)
        r = (y_obs[None, :] - mu)
        return -0.5*np.sum(np.log(2*np.pi*sigma2) + (r**2)/sigma2, axis=1)

    # ---- Fibroblast ----
    mu, vg = mu_all[:, IDX_FIB, obs], var_all[:, IDX_FIB, obs]
    ll += gauss_ll(y_fib_obs_N, y_fib_obs_std_N, mu, vg)

    # ---- Collagen ----
    mu, vg = mu_all[:, IDX_COL, obs], var_all[:, IDX_COL, obs]
    ll += gauss_ll(y_col_obs_N, y_col_obs_std_N, mu, vg)

    # ---- Macrophage ----
    mu, vg = mu_all[:, IDX_MAC, obs], var_all[:, IDX_MAC, obs]
    ll += gauss_ll(y_mac_obs_N, y_mac_obs_std_N, mu, vg)

    # ---- Capillary ----
    mu, vg = mu_all[:, IDX_CAP, obs], var_all[:, IDX_CAP, obs]
    ll += gauss_ll(y_cap_obs_N, y_cap_obs_std_N, mu, vg)

    # ---- Oxygen soft constraints ----
    mu = mu_all[:, IDX_OXY, :]
    vg = var_all[:, IDX_OXY, :]
    std_tot = np.sqrt(np.clip(vg, eps, None))

    # 1) Week-1 should be low (≤ 5 mmHg)
    t1 = week_to_idx[1]
    ll += norm.logcdf((OXY_MIN_W1_N - mu[:, t1]) / std_tot[:, t1])

    # 2) Cap only at the week with largest predicted mean
    t_star = np.argmax(mu, axis=1)
    mu_star  = mu[np.arange(N), t_star]
    std_star = std_tot[np.arange(N), t_star]
    ll += norm.logcdf((OXY_CAP_N - mu_star) / np.clip(std_star, eps, None))

    # ---- AF 5× peak ----
    mu_af  = mu_all[:, IDX_AF, :]
    var_af = var_all[:, IDX_AF, :]

    mu0  = np.maximum(mu_af[:, week_to_idx[0]], eps)
    var0 = np.maximum(var_af[:, week_to_idx[0]], 0.0)

    t_peak_idx = np.argmax(mu_af, axis=1)
    mu_peak  = np.maximum(mu_af[np.arange(N), t_peak_idx], eps)
    var_peak = np.maximum(var_af[np.arange(N), t_peak_idx], 0.0)

    R = np.log(mu_peak) - np.log(mu0)
    sigma_log2 = (var_peak / (mu_peak**2)) + (var0 / (mu0**2))

    sd_exp = 3.2
    mean_target = 5.0
    tau2_log = np.log1p((sd_exp / mean_target)**2)
    sigma_tot2 = np.clip(sigma_log2 + tau2_log, 1e-12, None)

    ll += -0.5 * ((R - np.log(mean_target))**2) / sigma_tot2 - 0.5 * np.log(2 * np.pi * sigma_tot2)

    # ============================================================
    # NEW: Peak-time ordering constraints (one-sided CDF at 0)
    # ============================================================

    weeks_arr = np.asarray(weeks, dtype=float)  # length T (actual week values)

    # peak times in weeks using GP mean trajectories
    tpk_fib = weeks_arr[np.argmax(mu_all[:, IDX_FIB, :], axis=1)]
    tpk_mac = weeks_arr[np.argmax(mu_all[:, IDX_MAC, :], axis=1)]
    tpk_cap = weeks_arr[np.argmax(mu_all[:, IDX_CAP, :], axis=1)]
    tpk_af  = weeks_arr[np.argmax(mu_all[:, IDX_AF,  :], axis=1)]

    # We want: A before B  =>  (tA - tB) <= 0
    dt_mac_fib = tpk_mac - tpk_fib
    dt_mac_cap = tpk_mac - tpk_cap
    dt_af_cap  = tpk_af  - tpk_cap

    def safe_logcdf(z, floor=1e-300):
        """
        Numerically safe log Φ(z).
        - For extreme negative z, norm.cdf(z) underflows to 0 -> log(0) = -inf.
        - We clip to a tiny floor so you get a large negative number instead of -inf.
        """
        return np.log(np.clip(norm.cdf(z), floor, 1.0))

    # one-sided soft inequality: log Φ( (0 - dt)/sigma_peak )
    ll += safe_logcdf((0.0 - dt_mac_fib) / sigma_peak)  # mac before fib
    ll += safe_logcdf((0.0 - dt_mac_cap) / sigma_peak)  # mac before cap
    ll += safe_logcdf((0.0 - dt_af_cap)  / sigma_peak)  # AF before cap

    return ll  # (N,)


import numpy as np

LOG_FLOOR = 1e-300  # prevents log(0)

def log_prior_batch(theta_phys_batch):
    """
    theta_phys_batch: (N,D) in PHYSICAL units
    returns: (N,) log prior under independent KDE priors in SCALED space
             using precomputed lookup tables (grid_z/grid_logpdf) + interpolation.
    """
    theta_phys_batch = np.asarray(theta_phys_batch, float)
    if theta_phys_batch.ndim == 1:
        theta_phys_batch = theta_phys_batch.reshape(1, -1)

    # Convert to scaled space
    theta_s = to_scaled_theta(theta_phys_batch)  # (N,D)
    N, D = theta_s.shape

    # Use the lookup-table dimension, not kdes_scaled
    if D != len(grid_z):
        raise ValueError(f"theta has D={D} but expected {len(grid_z)}")

    # Hard support bounds in scaled space
    in_bounds = np.all((theta_s >= low_scaled) & (theta_s <= high_scaled), axis=1)

    lp = np.full(N, -np.inf, dtype=float)
    if not np.any(in_bounds):
        return lp

    theta_ok = theta_s[in_bounds, :]  # (N_ok, D)
    lp_ok = np.zeros(theta_ok.shape[0], dtype=float)

    # Interpolate logpdf for each dimension (cheap)
    log_floor = np.log(LOG_FLOOR)
    for j in range(D):
        zgrid = grid_z[j]
        lgrid = grid_logpdf[j]

        x = theta_ok[:, j]

        # If within hard bounds but outside grid, assign LOG_FLOOR (like your scalar version)
        outside = (x < zgrid[0]) | (x > zgrid[-1])

        # np.interp is vectorized
        lp_dim = np.interp(x, zgrid, lgrid)
        lp_dim[outside] = log_floor

        lp_ok += lp_dim

    lp[in_bounds] = lp_ok
    return lp


def log_prior(theta_phys_1d):
    return float(log_prior_batch(np.asarray(theta_phys_1d, float).reshape(1, -1))[0])


def log_posterior_batch(theta_phys_batch):
    lp = log_prior_batch(theta_phys_batch)
    ll = log_likelihood_total_batch(theta_phys_batch)
    out = lp + ll
    out[~np.isfinite(out)] = -np.inf
    return out


def log_posterior(theta_phys_1d):
    return float(log_posterior_batch(np.asarray(theta_phys_1d, float).reshape(1, -1))[0])


# --- helper transforms (same as yours) ---
def to_scaled_theta(theta_phys):
    arr = np.asarray(theta_phys, float)
    if arr.ndim == 1:
        return scaler_params.transform(arr.reshape(1, -1))
    return scaler_params.transform(arr)

def to_phys_theta(theta_scaled):
    arr = np.asarray(theta_scaled, float)
    if arr.ndim == 1:
        return scaler_params.inverse_transform(arr.reshape(1, -1))
    return scaler_params.inverse_transform(arr)

In [6]:
import os
import numpy as np
import emcee
from emcee import moves
from emcee.autocorr import AutocorrError
from emcee.backends import HDFBackend

# =========================
# 1) Convenience functions
# =========================

def sample_physical_starts(n_walkers, param_bounds, eps_frac=1e-3, rng=None):
    rng = np.random.default_rng(rng)
    lows  = param_bounds[:, 0]
    highs = param_bounds[:, 1]
    width = highs - lows
    # small epsilon margin to avoid sitting exactly on bounds
    lows_eps  = lows  + eps_frac * width
    highs_eps = highs - eps_frac * width
    return rng.uniform(lows_eps, highs_eps, size=(n_walkers, len(lows)))

# Start walkers uniformly inside your SCALED bounds:
def sample_scaled_starts(n_walkers, low_scaled, high_scaled, eps_frac=1e-3, rng=None):
    rng = np.random.default_rng(rng)
    width = high_scaled - low_scaled
    lo = low_scaled + eps_frac * width
    hi = high_scaled - eps_frac * width
    return rng.uniform(lo, hi, size=(n_walkers, low_scaled.size))

# Target that emcee sees: takes SCALED theta, maps to PHYSICAL for your likelihoods
def log_posterior_from_scaled(theta_scaled_batch):
    theta_scaled_batch = np.asarray(theta_scaled_batch, float)
    # Map to physical once per chunk
    theta_phys_batch = to_phys_theta(theta_scaled_batch)
    # Keep using your existing functions that expect PHYSICAL:
    return log_posterior_batch(theta_phys_batch)


# =========================
# 2) Sampler settings
# =========================
D = 14
n_walkers = 4 * D

nburn = 2000
burn_chunk = 100

max_steps  = 60000
prod_chunk = 100

progress_txt = "mcmc_progress_log13.txt"

backend_file = "emcee_backend_log13.h5"
backend_name = "run13"

if os.path.exists(backend_file):
    print("♻️ Backend file found. Resuming.")
    backend = HDFBackend(backend_file, name=backend_name)
else:
    print("🆕 Backend file not found. Creating new backend.")
    backend = HDFBackend(backend_file, name=backend_name)
    backend.reset(n_walkers, D)


# We will store whether burn-in is done inside the progress log (simple & robust)
def burnin_completed_from_log(path):
    if not os.path.exists(path):
        return False
    try:
        with open(path, "r") as f:
            txt = f.read()
        return ("=== BURN-IN COMPLETE ===" in txt)
    except Exception:
        return False

burnin_done = burnin_completed_from_log(progress_txt)

# If backend has samples but burn-in isn't marked complete, we assume we're resuming burn-in.
if backend.iteration > 0:
    # Resume state from last saved sample
    last = backend.get_last_sample()
    state = last.coords  # (n_walkers, D) in SCALED space
else:
    # Start fresh
    p0_scaled = sample_scaled_starts(n_walkers, low_scaled, high_scaled, rng=42).astype(np.float64)
    state = p0_scaled
    # Only reset backend if starting fresh
    backend.reset(n_walkers, D)


# =========================
# 4) Build sampler (MUST match between resumes)
# =========================
sampler = emcee.EnsembleSampler(
    n_walkers, D,
    log_posterior_from_scaled,
    vectorize=True,
    moves=[
        (moves.DEMove(),            0.6),
        (moves.DESnookerMove(),     0.3),
        (moves.StretchMove(a=3.0),  0.1),
    ],
    backend=backend,  # <-- critical: autosaves
)


# =========================
# 5) Helper to log progress
# =========================
def log(msg, also_print=True):
    if also_print:
        print(msg)
    with open(progress_txt, "a") as f:
        f.write(msg + "\n")
        f.flush()


# =========================
# 6) Run (interrupt-safe)
# =========================
try:
    # ---------- Burn-in ----------
    if not burnin_done:
        log("=== Starting burn-in ===")
        # If resuming burn-in, backend.iteration might already be >0.
        # We'll run until total burn-in steps in backend reach nburn.
        while backend.iteration < nburn:
            remaining = nburn - backend.iteration
            this_chunk = min(burn_chunk, remaining)

            state = sampler.run_mcmc(state, this_chunk, progress=False)
            acc = float(np.mean(sampler.acceptance_fraction))
            log(f"  Burn-in progress: {backend.iteration}/{nburn} steps | acc={acc:.3f}")

        # Burn-in complete: mark it in the log, then reset backend for production
        log("=== BURN-IN COMPLETE ===")
        log("Resetting backend for production (burn-in samples cleared).")

        # IMPORTANT: keep the final burn-in state as the starting point for production
        backend.reset(n_walkers, D)
        sampler.reset()
        burnin_done = True
        # state remains the last returned coords from run_mcmc (good)

    # ---------- Production ----------
    log("=== Starting production run (adaptive τ check) ===")
    while backend.iteration < max_steps:
        remaining = max_steps - backend.iteration
        this_chunk = min(prod_chunk, remaining)

        state = sampler.run_mcmc(state, this_chunk, progress=False)

        acc = float(np.mean(sampler.acceptance_fraction))
        log(f"  Production progress: {backend.iteration}/{max_steps} steps | acc={acc:.3f}")

        # try estimating τ and stopping early if N >= 50 * tau_max
        try:
            tau = sampler.get_autocorr_time(tol=0)
            tau_max = float(np.max(tau))
            N = sampler.iteration  # == backend.iteration here

            log(f"    τ_max≈{tau_max:.1f}; N/τ_max≈{N/tau_max:.1f}; criterion: N >= {50*tau_max:.1f}")

            if N >= 50 * tau_max:
                log("    Convergence criterion satisfied (N ≥ 50×τ_max). Stopping early.")
                break

        except AutocorrError:
            log("    τ not reliable yet (AutocorrError). Continuing...")

except KeyboardInterrupt:
    # This is what happens when you hit the stop button / interrupt the cell
    log("🛑 Interrupted by user. Backend is saved; rerun this cell to continue.", also_print=True)


# =========================
# 7) Collect results (from backend)
# =========================
# NOTE: This will load whatever has been saved so far (even if interrupted).
chain_full = backend.get_chain()          # (n_steps, n_walkers, D)
logprob_full = backend.get_log_prob()     # (n_steps, n_walkers)

samples_scaled = chain_full.reshape(-1, D)
samples_phys = to_phys_theta(samples_scaled)
logprob = logprob_full.reshape(-1)

acc_frac = float(np.mean(sampler.acceptance_fraction))
print(f"Acceptance fraction ~ {acc_frac:.2f}")
print("Posterior mean (phys):", np.mean(samples_phys, axis=0))
print("Posterior median (phys):", np.median(samples_phys, axis=0))

print(f"✅ Backend saved at: {backend_file} (name='{backend_name}') | steps saved: {chain_full.shape[0]}")

🆕 Backend file not found. Creating new backend.
=== Starting burn-in ===
  Burn-in progress: 100/2000 steps | acc=0.084
  Burn-in progress: 200/2000 steps | acc=0.079
  Burn-in progress: 300/2000 steps | acc=0.079
  Burn-in progress: 400/2000 steps | acc=0.080
  Burn-in progress: 500/2000 steps | acc=0.080
  Burn-in progress: 600/2000 steps | acc=0.081
  Burn-in progress: 700/2000 steps | acc=0.080
  Burn-in progress: 800/2000 steps | acc=0.081
  Burn-in progress: 900/2000 steps | acc=0.082
  Burn-in progress: 1000/2000 steps | acc=0.082
  Burn-in progress: 1100/2000 steps | acc=0.082
  Burn-in progress: 1200/2000 steps | acc=0.082
  Burn-in progress: 1300/2000 steps | acc=0.082
  Burn-in progress: 1400/2000 steps | acc=0.083
  Burn-in progress: 1500/2000 steps | acc=0.083
  Burn-in progress: 1600/2000 steps | acc=0.082
  Burn-in progress: 1700/2000 steps | acc=0.083
  Burn-in progress: 1800/2000 steps | acc=0.083
  Burn-in progress: 1900/2000 steps | acc=0.082
  Burn-in progress: 2000

    τ_max≈266.4; N/τ_max≈25.1; criterion: N >= 13322.1
  Production progress: 6800/60000 steps | acc=0.079
    τ_max≈267.1; N/τ_max≈25.5; criterion: N >= 13354.5
  Production progress: 6900/60000 steps | acc=0.079
    τ_max≈267.6; N/τ_max≈25.8; criterion: N >= 13377.9
  Production progress: 7000/60000 steps | acc=0.079
    τ_max≈269.2; N/τ_max≈26.0; criterion: N >= 13458.3
  Production progress: 7100/60000 steps | acc=0.079
    τ_max≈271.0; N/τ_max≈26.2; criterion: N >= 13550.8
  Production progress: 7200/60000 steps | acc=0.079
    τ_max≈270.9; N/τ_max≈26.6; criterion: N >= 13547.0
  Production progress: 7300/60000 steps | acc=0.079
    τ_max≈269.8; N/τ_max≈27.1; criterion: N >= 13488.8
  Production progress: 7400/60000 steps | acc=0.079
    τ_max≈268.1; N/τ_max≈27.6; criterion: N >= 13407.1
  Production progress: 7500/60000 steps | acc=0.080
    τ_max≈267.4; N/τ_max≈28.0; criterion: N >= 13371.7
  Production progress: 7600/60000 steps | acc=0.080
    τ_max≈268.1; N/τ_max≈28.3; criter

Acceptance fraction ~ 0.08
Posterior mean (phys): [4.23726800e-02 5.58489666e-02 2.73075757e+00 8.69925579e+02
 4.20542495e-12 5.42545292e-05 4.88332158e-11 4.22836320e-06
 1.65185616e-02 4.12991538e+03 1.48922298e-02 1.21189164e+04
 1.76237380e-02 1.69491543e-07]
Posterior median (phys): [4.35131236e-02 5.71307932e-02 2.70858065e+00 8.12503082e+02
 3.93642364e-12 5.47690789e-05 4.85554443e-11 4.29720001e-06
 1.62684362e-02 3.48357758e+03 1.28446763e-02 1.16629088e+04
 1.76605904e-02 1.62404538e-07]
✅ Backend saved at: emcee_backend_log13.h5 (name='run13') | steps saved: 14300


In [7]:
import torch, joblib, pickle

torch.save(
    {"model": model.state_dict(), "likelihood": likelihood.state_dict()},
    "gp_state13_full.pt"
)

joblib.dump(scaler_params, "scaler_params13_full.joblib")
joblib.dump(scalers_Y, "scalers_Y13_full.joblib")

with open("gp_meta13_full.pkl", "wb") as f:
    pickle.dump({"weeks": weeks, "T": T, "num_species": num_species, "Q": Q}, f)

print("✅ Saved GP and scalers.")

import numpy as np
import pickle

# Flatten the chains (this matches what you'd analyze later)
samples_scaled = sampler.get_chain(flat=True)
logprob      = sampler.get_log_prob(flat=True)

# Convert a copy to PHYSICAL units for downstream plots/reports
samples_phys   = to_phys_theta(samples_scaled)         # (N, D) in physical units

# Or, if you want the full unflattened chain for diagnostics:
chain_full   = sampler.get_chain()         # (n_steps, n_walkers, n_dim)
logprob_full = sampler.get_log_prob()      # (n_steps, n_walkers)

np.savez(
    "mcmc_samples_logprob13_full.npz",
    samples_scaled=samples_scaled,
    samples_phys=samples_phys,
    logprob=logprob,
    chain_full=chain_full,
    logprob_full=logprob_full
)

with open("sampler_state13_full.pkl", "wb") as f:
    pickle.dump(sampler, f)

print("✅ Saved MCMC samples and sampler state.")

✅ Saved GP and scalers.
✅ Saved MCMC samples and sampler state.
